# Image Data Analysis — Cat Breed Classification (EDA + KNN)

This notebook extends the original pipeline with proper exploratory data analysis (EDA)
before jumping into KNN. The goal is to understand the dataset (sizes, brightness, blur,
class balance) and then compare **different feature representations** (RGB, grayscale, HOG)
and **different KNN hyperparameters** (K, distance metric) in a systematic way — instead of
running KNN once on raw pixels.

**Rule for this notebook:** all tuning (choice of K, metric, feature type) happens using the
training set only (via cross-validation or a held-out validation split). The test set is
touched exactly once, at the end, for the final reported accuracy.

## 1. Load dataset and collect per-image metadata

In [ ]:
from pathlib import Path
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

dataset_path = Path("./dataset/cats")
size = 64

classes = sorted([folder.name for folder in dataset_path.iterdir() if folder.is_dir()])

records = []      # one row per image: metadata used for EDA
images_bgr = []    # raw (resized) BGR images, kept in memory for feature extraction later
labels = []

for label, d_class in enumerate(classes):
    directory = dataset_path / d_class

    for image_path in directory.iterdir():
        img = cv2.imread(str(image_path))
        if img is None:
            continue

        h, w = img.shape[:2]

        gray_full = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        brightness = gray_full.mean()
        blur = cv2.Laplacian(gray_full, cv2.CV_64F).var()

        img_resized = cv2.resize(img, (size, size))

        records.append({
            "path": str(image_path),
            "class": d_class,
            "label": label,
            "width": w,
            "height": h,
            "aspect_ratio": w / h,
            "brightness": brightness,
            "blur": blur,
        })
        images_bgr.append(img_resized)
        labels.append(label)

df = pd.DataFrame(records)
images_bgr = np.array(images_bgr)
labels = np.array(labels)

print("Classes:", classes)
print("Total images loaded:", len(df))
df.head()

## 2. Class distribution

Check for class imbalance before doing anything else.

In [ ]:
class_counts = df["class"].value_counts().reindex(classes)
print(class_counts)

plt.figure(figsize=(7, 4))
class_counts.plot(kind="bar", color="#4C72B0")
plt.title("Images per class")
plt.ylabel("Count")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

If the bars are roughly equal height, class imbalance is not a concern and accuracy is a
fair metric. If one class dominates, prefer balanced accuracy / macro-F1 and consider
stratified sampling.

## 3. Image size distribution

Before blindly resizing everything to 64×64, see how varied the original dimensions are.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].scatter(df["width"], df["height"], c=df["label"], cmap="tab10", alpha=0.6, s=15)
axes[0].set_xlabel("Width (px)")
axes[0].set_ylabel("Height (px)")
axes[0].set_title("Original image dimensions")

axes[1].hist(df["aspect_ratio"], bins=30, color="#55A868")
axes[1].set_xlabel("Aspect ratio (w / h)")
axes[1].set_title("Aspect ratio distribution")

plt.tight_layout()
plt.show()

print(df[["width", "height", "aspect_ratio"]].describe())

A wide spread of resolutions and aspect ratios means resizing to a fixed square (64×64)
distorts some images more than others. Worth noting as a limitation, even if you keep the
simple resize for this assignment.

## 4. Brightness analysis

If one breed's photos are systematically brighter or darker than another's, a pixel-based classifier like KNN may partly be learning lighting/background instead of the cat.

In [ ]:
plt.figure(figsize=(8, 4))
df.boxplot(column="brightness", by="class", grid=False)
plt.title("Brightness by class")
plt.suptitle("")
plt.ylabel("Mean pixel intensity (0-255)")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

df.groupby("class")["brightness"].describe()[["mean", "std", "min", "max"]]

## 5. Blur analysis

Blurry images carry less usable signal. The variance of the Laplacian is a common sharpness proxy — lower values indicate blurrier images.

In [ ]:
plt.figure(figsize=(8, 4))
df.boxplot(column="blur", by="class", grid=False)
plt.title("Blurriness (Laplacian variance) by class")
plt.suptitle("")
plt.ylabel("Laplacian variance (higher = sharper)")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

blur_threshold = df["blur"].quantile(0.10)
very_blurry = df[df["blur"] <= blur_threshold]
print(f"Flagging bottom 10% sharpest-cutoff (blur <= {blur_threshold:.1f}): {len(very_blurry)} images")
very_blurry[["path", "class", "blur"]].head(10)

These flagged images are candidates to inspect manually and possibly exclude — but do this
**before** the train/test split is fixed, and document the decision rather than silently
dropping test-set images after seeing results.

## 6. Sample images per class

A quick visual sanity check: are cats centered/cropped, and do backgrounds vary a lot?

In [ ]:
fig, axes = plt.subplots(len(classes), 5, figsize=(12, 2.4 * len(classes)))

for row, cls in enumerate(classes):
    sample_paths = df[df["class"] == cls]["path"].sample(min(5, (df["class"] == cls).sum()), random_state=0)
    for col, p in enumerate(sample_paths):
        img = cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB)
        ax = axes[row, col] if len(classes) > 1 else axes[col]
        ax.imshow(img)
        ax.axis("off")
        if col == 0:
            ax.set_ylabel(cls, fontsize=10)
    axes[row, 0].set_title(cls, loc="left", fontsize=11)

plt.tight_layout()
plt.show()

## 7. Feature representations

Compare three ways of turning an image into a feature vector for KNN:

- **RGB pixels** — the original approach (64×64×3 = 12,288 features)
- **Grayscale pixels** — drops color, keeps intensity (64×64 = 4,096 features)
- **HOG (Histogram of Oriented Gradients)** — encodes edge/shape structure instead of raw color

All three are built from the *same* set of images so the comparison is apples-to-apples.

In [ ]:
from skimage.feature import hog

def to_rgb_features(images_bgr):
    return np.array([img.flatten() / 255.0 for img in images_bgr])

def to_gray_features(images_bgr):
    feats = []
    for img in images_bgr:
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        feats.append(gray.flatten() / 255.0)
    return np.array(feats)

def to_hog_features(images_bgr):
    feats = []
    for img in images_bgr:
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        feat = hog(
            gray,
            orientations=9,
            pixels_per_cell=(8, 8),
            cells_per_block=(2, 2),
            block_norm="L2-Hys",
        )
        feats.append(feat)
    return np.array(feats)

X_rgb = to_rgb_features(images_bgr)
X_gray = to_gray_features(images_bgr)
X_hog = to_hog_features(images_bgr)

print("RGB features:      ", X_rgb.shape)
print("Grayscale features:", X_gray.shape)
print("HOG features:      ", X_hog.shape)

## 8. Train/test split

Split **once**, stratified by class, and reuse the exact same split for every feature set and hyperparameter combination below. This keeps comparisons fair and keeps the test set untouched until the final evaluation.

In [ ]:
from sklearn.model_selection import train_test_split

def split(X, y, seed=42):
    return train_test_split(X, y, test_size=0.2, stratify=y, random_state=seed)

splits = {
    "RGB": split(X_rgb, labels),
    "Grayscale": split(X_gray, labels),
    "HOG": split(X_hog, labels),
}

for name, (Xtr, Xte, ytr, yte) in splits.items():
    print(f"{name:10s} train={Xtr.shape[0]:3d}  test={Xte.shape[0]:3d}  feature_dim={Xtr.shape[1]}")

## 9. KNN experiments: feature type × K × distance metric

Sweep across every combination and record accuracy, instead of running KNN once with K=5.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

k_values = [1, 3, 5, 7, 9, 11, 15]
metrics = ["euclidean", "manhattan"]

results = []

for feature_name, (Xtr, Xte, ytr, yte) in splits.items():
    for metric in metrics:
        for k in k_values:
            knn = KNeighborsClassifier(n_neighbors=k, metric=metric)
            knn.fit(Xtr, ytr)
            acc = knn.score(Xte, yte)
            results.append({
                "features": feature_name,
                "metric": metric,
                "k": k,
                "accuracy": acc,
            })

results_df = pd.DataFrame(results)
results_df.sort_values("accuracy", ascending=False).head(15)

## 10. Compare feature representations visually

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)

for ax, metric in zip(axes, metrics):
    subset = results_df[results_df["metric"] == metric]
    for feature_name in splits.keys():
        line = subset[subset["features"] == feature_name].sort_values("k")
        ax.plot(line["k"], line["accuracy"], marker="o", label=feature_name)
    ax.set_title(f"Distance metric: {metric}")
    ax.set_xlabel("K (number of neighbors)")
    ax.set_ylabel("Test accuracy")
    ax.legend()
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 11. Best configuration

Pick the single best row from the sweep — this is the only number that should be reported as "final" test accuracy.

In [ ]:
best = results_df.loc[results_df["accuracy"].idxmax()]
print("Best configuration:")
print(best)

pivot = results_df.pivot_table(index=["features", "metric"], columns="k", values="accuracy")
pivot

### Notes for the write-up

- Raw RGB pixels force KNN to compare exact colors at exact pixel positions, so any shift in
  pose, lighting, or background changes the distance a lot even for the same breed.
- Grayscale removes color noise but keeps the same "compare raw pixels" weakness.
- HOG describes edge orientation patterns instead of raw intensity, which tends to generalize
  better to pose/lighting variation for traditional (non-deep-learning) classifiers — but it
  isn't guaranteed to win on every dataset, which is exactly why the sweep above matters more
  than assuming an outcome.
- If the flagged blurry images in step 5 turn out to concentrate in one class, mention that as
  a confound rather than only reporting the accuracy number.
